# **A. Setup**

  ### A.I. Import Libary


In [ ]:
## Import
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Check Python Version
torch.__version__
# Setup Device Agnostic Code
device = "cuda" if torch.cuda.is_available() else "cpu"
#device

In [ ]:
# Get torchinfo, install it if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

# Try to import the going_modular directory, download it from GitHub if it doesn't work
try:
    from going_modular.going_modular import data_setup, engine
except:
    # Get the going_modular scripts
    print("[INFO] Couldn't find going_modular scripts... downloading them from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !rm -rf pytorch-deep-learning
    from going_modular.going_modular import data_setup, engine

### A.II Data Setup

In [ ]:
## 1.1 IMPORT DATA
import os
import zipfile
from pathlib import Path
import requests

# Setup path to data folder
data_path = Path("data/full")
image_path = data_path / "full"


# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# if the image folder doesn't exist, create one
if not image_path.is_dir():
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)
else:
    print(f"{image_path} directory already exists")


# Define the path to your data in Google Drive
gdrive_zip_path = '/content/drive/MyDrive/Thesis/Data/compare_data/full/full.zip'

# Define the destination path in the Colab environment
colab_zip_path = data_path / "full.zip"
colab_data_destination_path = data_path # Copy the directory into the 'data' folder

if os.path.exists(gdrive_zip_path):
    print(f"Found zip file at {gdrive_zip_path}. Copying and unzipping...")
    # Copy the zip file from Google Drive to the Colab environment
    # Use -f to force overwrite if the file already exists in the destination
    !cp -f '{gdrive_zip_path}' '{colab_zip_path}'

    # Unzip data to the image_path directory
    if os.path.exists(colab_zip_path):
        print(f"Unzipping {colab_zip_path} to {image_path}...")
        with zipfile.ZipFile(colab_zip_path, "r") as zip_ref:
            zip_ref.extractall(image_path)
        print("Unzipping complete.")
    else:
        print(f"Error: Zip file not found in Colab environment at {colab_zip_path} after copying.")

# If the zip file is not found, try to copy the data directory
elif os.path.exists(gdrive_data_path):
    print(f"Zip file not found. Found data directory at {gdrive_data_path}. Copying directory...")
    # Copy the data directory from Google Drive to the Colab environment
    # Use -r for recursive copying and -f to force overwrite if the directory already exists
    !cp -r -f '{gdrive_data_path}' '{colab_data_destination_path}'
    print("Copying complete.")
else:
    print(f"Error: Data not found in Google Drive at either {gdrive_zip_path} or {gdrive_data_path}")


In [ ]:
# Setup Directories
train_dir = image_path / "train"
test_dir = image_path / "test"

#train_dir, test_dir

# **B. Model**

## B.I. Pre-train

In [ ]:
# 1.2 SETUP PRETRAINED MODEL
weights = torchvision.models.MobileNet_V2_Weights.DEFAULT

In [ ]:
# Get the transform used to create pretrained weights
auto_transforms= weights.transforms()

In [ ]:
# Create training and testing DataLoaders as well as get a list of class names
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                               test_dir=test_dir,
                                                                               transform=auto_transforms, # perform same data transforms on our own data as the pretrained model
                                                                               batch_size=32) # set mini-batch size to 32

In [ ]:
#Setup the model with pretrained weights and send it to the target device
model = torchvision.models.mobilenet_v2(weights=weights).to(device)

In [ ]:
# Freeze all base layers in the model (except the classifier) by setting requires_grad=False
for param in model.parameters():
    param.requires_grad = False

In [ ]:
# Set the manual seeds
#torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Get the length of class_names (one output unit for each class)
output_shape = len(class_names)

# Recreate the classifier layer and seed it to the target device
model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1280,
                    out_features=output_shape, # same number of output units as our number of classes
                    bias=True)).to(device)

## B.II Train

In [ ]:
# Define loss and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Start the timer
from timeit import default_timer as timer
start_time = timer()

# Setup training and save the results
results = engine.train(model=model,
                       train_dataloader=train_dataloader,
                       test_dataloader=test_dataloader,
                       optimizer=optimizer,
                       loss_fn=loss_fn,
                       epochs=20,
                       device=device)

# End the timer and print out how long it took
end_time = timer()
print(f"[INFO] Total training time: {end_time-start_time:.3f} seconds")

# **C. Evaluation**

## C.I Loss and Accuracy

In [ ]:
# Get the plot_loss_curves() function from helper_functions.py
# from https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py
try:
    from helper_functions import plot_loss_curves
except:
    print("[INFO] Couldn't find helper_functions.py, downloading...")
    with open("helper_functions.py", "wb") as f:
        import requests
        request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
        f.write(request.content)
    from helper_functions import plot_loss_curves

# Plot the loss curves of our model
plot_loss_curves(results)

## C.II Confusion Matrix

##### C.II.1. Initialize

In [ ]:
# Training Data
all_train_images = []
all_train_labels = []
all_train_predictions = []
all_train_probs = []

model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    for data, labels in train_dataloader:
        data, labels = data.to(device), labels.to(device)
        outputs = model(data)
        probabilities = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)

        all_train_images.extend(data.cpu())
        all_train_labels.extend(labels.cpu().numpy())
        all_train_predictions.extend(predicted.cpu().numpy())
        all_train_probs.extend(probabilities.cpu().numpy())

In [ ]:
# Testing Data
all_test_images = []
all_test_labels = []
all_test_predictions = []
all_test_probs = []

model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    for data, labels in test_dataloader:
        data, labels = data.to(device), labels.to(device)
        outputs = model(data)
        probabilities = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)

        all_test_images.extend(data.cpu())
        all_test_labels.extend(labels.cpu().numpy())
        all_test_predictions.extend(predicted.cpu().numpy())
        all_test_probs.extend(probabilities.cpu().numpy())

##### C.II.2 Plot Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Plot Confusion Matrix for Testing Data
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1) # 1 row, 2 columns, 1st plot
cm_test = confusion_matrix(all_test_labels, all_test_predictions, normalize='true')  # Use previously collected test data
sns.heatmap(cm_test, annot=True, fmt=".2f", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Normalized Confusion Matrix (Testing Data)")

# Plot Confusion Matrix for Training Data
plt.subplot(1, 2, 2) # 1 row, 2 columns, 2nd plot
cm_train = confusion_matrix(all_train_labels, all_train_predictions, normalize='true') # Use previously collected training data
sns.heatmap(cm_train, annot=True, fmt=".2f", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Normalized Confusion Matrix (Training Data)")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Plot Confusion Matrix for Testing Data (Regular Counts)
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1) # 1 row, 2 columns, 1st plot
cm_test = confusion_matrix(all_test_labels, all_test_predictions)  # Regular counts
sns.heatmap(cm_test, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues") # Use fmt="d" for integers
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Testing Data - Counts)")

# Plot Confusion Matrix for Training Data (Regular Counts)
plt.subplot(1, 2, 2) # 1 row, 2 columns, 2nd plot
cm_train = confusion_matrix(all_train_labels, all_train_predictions) # Regular counts
sns.heatmap(cm_train, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues") # Use fmt="d" for integers
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Training Data - Counts)")

plt.tight_layout()
plt.show()

## C.III Histogram

In [ ]:
# Function to display Histogram
import seaborn as sns
def plot_confidence_histogram(probs, preds, labels):
    confidences = []
    correctness = []

    for i in range(len(labels)):
        confidence = probs[i][preds[i]]
        confidences.append(confidence)
        correctness.append(preds[i] == labels[i])

    sns.histplot(x=confidences, hue=correctness, bins=20, kde=True,
                 palette={True: "green", False: "red"})
    plt.xlabel("Model Confidence")
    plt.ylabel("Prediction Count")
    plt.grid(True)
    plt.show()

In [ ]:
# Plot the Hisogram of Train and Test Dataset
import matplotlib.pyplot as plt
import seaborn as sns


plt.figure(figsize=(8, 10)) # Adjusted figure size for vertical layout
# Plot Confidence Histogram for Testing Data
plt.subplot(2, 1, 1) # 2 rows, 1 column, 1st plot
plt.title("Confidence Histogram (Testing Data)")
plot_confidence_histogram(all_test_probs, all_test_predictions, all_test_labels) # Use previously collected test data



# Plot Confidence Histogram for Training Data
plt.figure(figsize=(8, 10)) # Adjusted figure size for vertical layout
plt.subplot(2, 1, 2) # 2 rows, 1 column, 2nd plot
plt.title("Confidence Histogram (Training Data)")
plot_confidence_histogram(all_train_probs, all_train_predictions, all_train_labels) # Use previously collected training data


plt.tight_layout()
plt.show()

## C.III Display Incorrect Predictions

##### C.III.1. Prepare

In [ ]:
# Create a Dataset include the image Path. Achieve by inheriting from ImageFolder and overriding the __getitem__ method.
from torchvision.datasets import ImageFolder
from typing import Tuple
import os
from pathlib import Path

class ImageFolderWithPaths(ImageFolder):
    """Custom ImageFolder that includes image paths."""
    def __getitem__(self, index: int) -> Tuple[torch.Tensor, int, str]:
        """
        Gets image and target from the dataset with image path.
        """
        # Get the original item (image and target) from the parent class
        original_tuple = super().__getitem__(index)
        image, label = original_tuple

        # Get the image path using the index
        path = self.paths[index]

        return image, label, path


In [ ]:
# Test Dataset with Path
from torchvision.datasets import ImageFolder
from typing import Tuple
import torch

class ImageFolderWithPaths(ImageFolder):
    """Custom ImageFolder that includes image paths."""
    def __getitem__(self, index: int) -> Tuple[torch.Tensor, int, str]:
        """
        Gets image and target from the dataset with image path.
        """
        # Get the original item (image and target) from the parent class
        # The ImageFolder class stores the paths in self.samples, which is a list of (image_path, class_index) tuples
        original_tuple = super().__getitem__(index)
        image, label = original_tuple

        # Get the image path using the index from self.samples
        path = self.samples[index][0]

        return image, label, path

# 1. Instantiate the ImageFolderWithPaths dataset for the test directory
test_dataset_with_paths = ImageFolderWithPaths(root=test_dir,
                                                 transform=auto_transforms)

# 2. Create a new DataLoader for the test dataset
test_dataloader_with_paths = torch.utils.data.DataLoader(test_dataset_with_paths,
                                                           batch_size=32,
                                                           shuffle=False) # No need to shuffle test data

# 3. Iterate through the new test DataLoader and collect data
all_test_images = []
all_test_labels = []
all_test_predictions = []
all_test_probs = []
all_test_image_paths = []

model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    for data, labels, paths in test_dataloader_with_paths:
        data, labels = data.to(device), labels.to(device)
        outputs = model(data)
        probabilities = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)

        all_test_images.extend(data.cpu())
        all_test_labels.extend(labels.cpu().numpy())
        all_test_predictions.extend(predicted.cpu().numpy())
        all_test_probs.extend(probabilities.cpu().numpy())
        all_test_image_paths.extend(paths) # Collect the image paths

In [ ]:
import random
import matplotlib.pyplot as plt
from math import ceil # Import ceil for subplot calculation
import os # Import os to get basename

def show_incorrect_predictions(images, labels, preds, probs, class_names, image_names=None, max_images=25):
    """
    Shows a random sample of incorrect predictions with their predicted and true labels,
    confidence, and image filename.
    """
    incorrect = [i for i in range(len(labels)) if labels[i] != preds[i]]

    # Print the total number of incorrect predictions
    print(f"Total incorrect predictions: {len(incorrect)}")

    # Create a list of tuples (original_index, image_name) for incorrect predictions
    incorrect_with_names = []
    if image_names:
        for i in incorrect:
            # Get the image name using the original index
            img_name = os.path.basename(image_names[i]) if i < len(image_names) else f"Image {i}"
            incorrect_with_names.append((i, img_name))
    else:
        # If image_names are not provided, just use the original index
        incorrect_with_names = [(i, f"Image {i}") for i in incorrect]


    random.shuffle(incorrect_with_names) # Shuffle the list of tuples (original_index, image_name)
    num_samples = min(len(incorrect_with_names), max_images)

    # Calculate the number of rows needed for 2 images per row
    num_rows = ceil(num_samples / 2)

    plt.figure(figsize=(15, num_rows * 4)) # Adjust figure size based on the number of rows
    for i in range(num_samples):
        original_idx, img_name = incorrect_with_names[i] # Unpack the original index and image name

        img_tensor = images[original_idx] # Use the original index to get the image tensor
        # Unnormalize the image tensor for plotting
        img = img_tensor.permute(1, 2, 0)
        img = img * torch.tensor([0.229, 0.224, 0.225]) + torch.tensor([0.485, 0.456, 0.406])
        img = img.clamp(0, 1).numpy()

        true_label = class_names[labels[original_idx]] # Use the original index to get the true label
        pred_label = class_names[preds[original_idx]] # Use the original index to get the predicted label
        prob = probs[original_idx][preds[original_idx]] * 100 # Use the original index to get the probability


        # Use ceil(num_samples / 2) for the number of rows and 2 for the number of columns
        plt.subplot(num_rows, 2, i + 1)
        plt.imshow(img)
        plt.title(f"❌ {pred_label} ({prob:.1f}%) | True: {true_label} | {img_name}") # Add image name to title
        plt.axis("off")
    plt.tight_layout()
    plt.show()


##### C.III.2. Data

In [ ]:
# 1. Instantiate the ImageFolderWithPaths dataset for the train directory
train_dataset_with_paths = ImageFolderWithPaths(root=train_dir,
                                                 transform=auto_transforms)

# 2. Create a new DataLoader for the train dataset
# Use shuffle=False here to maintain a consistent order for collecting image paths
train_dataloader_with_paths = torch.utils.data.DataLoader(train_dataset_with_paths,
                                                           batch_size=32,
                                                           shuffle=False) # No need to shuffle when collecting data for plotting

# 3. Iterate through the new train DataLoader and collect data and paths
all_train_images = []
all_train_labels = []
all_train_predictions = []
all_train_probs = []
all_train_image_paths = []

model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    for data, labels, paths in train_dataloader_with_paths:
        data, labels = data.to(device), labels.to(device)
        outputs = model(data)
        probabilities = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)

        all_train_images.extend(data.cpu())
        all_train_labels.extend(labels.cpu().numpy())
        all_train_predictions.extend(predicted.cpu().numpy())
        all_train_probs.extend(probabilities.cpu().numpy())
        all_train_image_paths.extend(paths) # Collect the image paths

##### C.III.3. Display

In [ ]:
# Show incorrect predictions on the Training set with image names
print("Incorrect Predictions on Training Data:")
show_incorrect_predictions(all_train_images,
                           all_train_labels,
                           all_train_predictions,
                           all_train_probs,
                           class_names,
                           image_names=all_train_image_paths)

In [ ]:
# Show incorrect predictions on the Testing set with image names
print("Incorrect Predictions on Testing Data:")
show_incorrect_predictions(all_test_images,
                           all_test_labels,
                           all_test_predictions,
                           all_test_probs,
                           class_names,
                           image_names=all_test_image_paths)

## C.IV Test Single Image

##### C.IV.1. Prepare the function

In [ ]:
# Prepare to input image in the dataset
from typing import List, Tuple

from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path


# 1. Take in a trained model, class names, image path, image size, a transform and target device
def pred_and_plot_image(model: torch.nn.Module,
                        image_path: str,
                        class_names: List[str],
                        image_size: Tuple[int, int] = (224, 224),
                        transform: torchvision.transforms = None,
                        device: torch.device=device):


    # 2. Open image
    img = Image.open(image_path)
    img_name= Path(image_path).name

    # 3. Create transformation for image (if one doesn't exist)
    if transform is not None:
        image_transform = transform
    else:
        image_transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    ### Predict on image ###

    # 4. Make sure the model is on the target device
    model.to(device)

    # 5. Turn on model evaluation mode and inference mode
    model.eval()
    with torch.inference_mode():
      # 6. Transform and add an extra dimension to image (model requires samples in [batch_size, color_channels, height, width])
      transformed_image = image_transform(img).unsqueeze(dim=0)

      # 7. Make a prediction on image with an extra dimension and send it to the target device
      target_image_pred = model(transformed_image.to(device))

    # 8. Convert logits -> prediction probabilities (using torch.softmax() for multi-class classification)
    target_image_pred_probs = torch.softmax(target_image_pred, dim=1)

    # 8.1 Extract the maximum probability
    max_prob = target_image_pred_probs.max()


    # 9. Convert prediction probabilities -> prediction labels
    target_image_pred_label = torch.argmax(target_image_pred_probs, dim=1)

    # 10. Plot image with predicted label and probability with condition of max_prob>80%
    plt.figure()
    plt.imshow(img)
    if max_prob < 0.8:
      plt.title(f"Pred: Working on it. {class_names[target_image_pred_label]} | Actual: {Path(image_path).parent.name} | Image: {img_name}")
      print("Class Probabilities:")
      for i, prob in enumerate(target_image_pred_probs[0]):
            print(f"{class_names[i]}: {prob:.3f}")
    else:
      plt.title(f"Pred: {class_names[target_image_pred_label]} | Actual: {Path(image_path).parent.name} | Prob: {target_image_pred_probs.max():.3f}")
    plt.axis(False);
    plt.show() # Ensure the plot is displayed

##### C.IV.2. Input the image and predict

In [ ]:
# # input random image
# from PIL import Image
# from google.colab import files
# import matplotlib.pyplot as plt

# uploaded = files.upload()

# # Get the filename of the uploaded image
# image_path = list(uploaded.keys())[0]

# # Use the pred_and_plot_image function to get the prediction
# pred_and_plot_image(model=model,
#                     image_path=image_path,
#                     class_names=class_names,
#                     transform=auto_transforms, # Use the same transforms as the model was trained with
#                     image_size=(224, 224),
#                     device=device)

### **E. Inference**

In [ ]:
# Calculate inference time for a single image
from timeit import default_timer as timer
from google.colab import files
from PIL import Image
import os

# Choose an image to test inference time on
# single_image_path = all_test_image_paths[0] # Original line

# Option 1: Manually specify a path to an image
# single_image_path = "/content/data/background/cardboard/test/your_image_name.jpg" # Replace with your image path

# Option 2: Upload an image file
uploaded = files.upload()
if uploaded:
    # Get the filename of the uploaded image
    single_image_path = list(uploaded.keys())[0]

    start_time = timer()
    pred_and_plot_image(model=model,
                        image_path=single_image_path,
                        class_names=class_names,
                        transform=auto_transforms,
                        image_size=(224, 224),
                        device=device)
    end_time = timer()

    print(f"Inference time for a single image: {end_time - start_time:.3f} seconds")
else:
    print("No file uploaded.")

In [ ]:
torch.save(model, "top&bottom.pth")


In [ ]:
#import torch
dummy_input = torch.randn(1, 3, 224, 224).to(device)
traced_model = torch.jit.trace(model, dummy_input)
model_save_path = "top&bottom.pt"
traced_model.save(model_save_path)
print(f"Model saved to {model_save_path}")